# Produits achetes ensemble : regles d'association

## Question metier
Quels produits sont achetes ensemble plus souvent que ce qu'on attendrait de leur popularite individuelle ? Une reponse peut suggerer des produits a recommander ensemble ou a presenter dans une offre groupee.

Cette analyse porte sur `order_products__prior.csv`. Elle explore des associations, pas des causes : acheter A ne signifie pas que A provoque l'achat de B.

## 1. Preparatifs

Executer d'abord `notebook/nettoyage_donnees.ipynb`. Pour garder le calcul accessible, on tire un echantillon aleatoire reproductible de commandes `prior`, puis on limite les candidats aux produits les plus presents dans cet echantillon.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

candidats = [Path.cwd(), *Path.cwd().parents]
PROJET = next((p for p in candidats if (p / "data" / "raw").is_dir()), None)
if PROJET is None:
    raise FileNotFoundError("Racine du projet introuvable : data/raw est absent.")

DONNEES = PROJET / "data" / "processed"
if not (DONNEES / "orders.csv").is_file() or not (DONNEES / "order_products__prior.csv").is_file():
    raise FileNotFoundError("Executer d'abord le notebook de nettoyage pour creer data/processed/.")

TAILLE_ECHANTILLON = 50_000
NOMBRE_PRODUITS_CANDIDATS = 50
SUPPORT_MINIMUM = 0.001  # au moins 50 paniers sur un echantillon de 50 000
CONFIANCE_MINIMUM = 0.20
GRAINE_ALEATOIRE = 42
print(f"Projet : {PROJET}")

Projet : C:\Users\mouft\Desktop\data\data_m2


## 2. Choisir 50 000 commandes historiques

La graine aleatoire fixe permet de retrouver le meme echantillon lors d'une nouvelle execution. La lecture par morceaux evite de charger `orders.csv` en entier.

In [2]:
morceaux_commandes_prior = []
for morceau in pd.read_csv(
    DONNEES / "orders.csv",
    usecols=["order_id", "eval_set"],
    chunksize=500_000,
):
    morceaux_commandes_prior.append(
        morceau.loc[morceau["eval_set"].eq("prior"), "order_id"].to_numpy()
    )

ids_prior = np.concatenate(morceaux_commandes_prior)
if len(ids_prior) < TAILLE_ECHANTILLON:
    raise ValueError("Il n'y a pas assez de commandes prior pour l'echantillon demande.")

rng = np.random.default_rng(GRAINE_ALEATOIRE)
ids_echantillon = rng.choice(ids_prior, size=TAILLE_ECHANTILLON, replace=False)
ids_echantillon_set = set(ids_echantillon.tolist())
print(f"Commandes prior disponibles : {len(ids_prior):,}")
print(f"Commandes selectionnees : {len(ids_echantillon):,}")

Commandes prior disponibles : 3,214,874
Commandes selectionnees : 50,000


## 3. Recuperer les produits de ces paniers

On parcourt le gros fichier `prior`, mais on ne conserve que les lignes des commandes selectionnees.

In [3]:
morceaux_paniers = []
lignes_lues = 0
for numero_morceau, morceau in enumerate(pd.read_csv(
    DONNEES / "order_products__prior.csv",
    usecols=["order_id", "product_id"],
    chunksize=500_000,
)):
    lignes_lues += len(morceau)
    selection = morceau.loc[
        morceau["order_id"].isin(ids_echantillon_set),
        ["order_id", "product_id"],
    ]
    if not selection.empty:
        morceaux_paniers.append(selection)
    if (numero_morceau + 1) % 10 == 0:
        print(f"{lignes_lues:,} lignes lues...")

lignes_echantillon = pd.concat(morceaux_paniers, ignore_index=True)
print(f"Lignes de produits conservees : {len(lignes_echantillon):,}")
display(lignes_echantillon.head())

5,000,000 lignes lues...


10,000,000 lignes lues...


15,000,000 lignes lues...


20,000,000 lignes lues...


25,000,000 lignes lues...


30,000,000 lignes lues...


Lignes de produits conservees : 501,917


,order_id,product_id
0,74,46667
1,74,14129
2,74,11136
3,74,27247
4,74,17902


## 4. Construire une matrice de paniers

FP-Growth a besoin d'une representation indiquant si chaque produit est present ou absent dans chaque panier. On limite d'abord l'analyse aux 50 produits les plus frequents de l'echantillon : cela reduit le calcul et garde l'exemple lisible. Le denominateur du support reste l'ensemble des 50 000 commandes selectionnees.

In [4]:
produits_candidats = (
    lignes_echantillon["product_id"]
    .value_counts()
    .head(NOMBRE_PRODUITS_CANDIDATS)
    .index
    .tolist()
)
lignes_candidates = lignes_echantillon[
    lignes_echantillon["product_id"].isin(produits_candidats)
]
produits_par_commande = (
    lignes_candidates.groupby("order_id")["product_id"]
    .agg(lambda valeurs: sorted(set(valeurs)))
    .to_dict()
)
paniers = [produits_par_commande.get(int(order_id), []) for order_id in ids_echantillon]

encodeur = TransactionEncoder()
matrice = encodeur.fit(paniers).transform(paniers)
paniers_binaires = pd.DataFrame(matrice, columns=encodeur.columns_)
print(f"Matrice : {paniers_binaires.shape[0]:,} paniers x {paniers_binaires.shape[1]} produits")
print(f"Produits candidats retenus : {len(produits_candidats)}")

Matrice : 50,000 paniers x 50 produits
Produits candidats retenus : 50


## 5. Extraire les paires et calculer les regles

On limite les combinaisons aux paires (`max_len=2`). Le support minimum de 0,001 signifie qu'une paire doit apparaitre dans au moins 50 paniers de l'echantillon. Le lift superieur a 1 indique une association positive dans cet echantillon.

In [5]:
ensembles_frequents = fpgrowth(
    paniers_binaires,
    min_support=SUPPORT_MINIMUM,
    use_colnames=True,
    max_len=2,
)
paires_frequentes = ensembles_frequents[
    ensembles_frequents["itemsets"].map(len).eq(2)
]
print(f"Paires qui atteignent le support minimum : {len(paires_frequentes)}")

if paires_frequentes.empty:
    raise ValueError(
        "Aucune paire n'atteint le support choisi. Il faut augmenter l'echantillon "
        "ou diminuer SUPPORT_MINIMUM, puis relancer l'analyse."
    )

regles = association_rules(
    ensembles_frequents,
    metric="confidence",
    min_threshold=CONFIANCE_MINIMUM,
)
regles = regles.loc[regles["lift"].gt(1)].copy()
print(f"Regles avec confiance >= {CONFIANCE_MINIMUM:.0%} et lift > 1 : {len(regles)}")

Paires qui atteignent le support minimum : 791
Regles avec confiance >= 20% et lift > 1 : 59


## 6. Lire les meilleures regles

- **Support** : proportion de tous les paniers echantillonnes qui contiennent les deux produits.
- **Confiance A vers B** : parmi les paniers qui contiennent A, proportion qui contient aussi B.
- **Lift** : compare cette confiance a la frequence generale de B. Un lift de 1,5 signifie que B apparait 1,5 fois plus souvent chez les acheteurs de A que dans les paniers en general.

In [6]:
noms_produits = pd.read_csv(DONNEES / "products.csv").set_index("product_id")["product_name"].to_dict()

def nom_unique_article(ensemble):
    identifiant = next(iter(ensemble))
    return noms_produits.get(identifiant, f"Produit {identifiant}")

regles["produit_A"] = regles["antecedents"].map(nom_unique_article)
regles["produit_B"] = regles["consequents"].map(nom_unique_article)
regles_a_lire = (
    regles.sort_values(["lift", "support"], ascending=False)
    [["produit_A", "produit_B", "support", "confidence", "lift"]]
    .head(15)
    .copy()
)
regles_a_lire[["support", "confidence", "lift"]] = regles_a_lire[["support", "confidence", "lift"]].round(3)
display(regles_a_lire.reset_index(drop=True))

,produit_A,produit_B,support,confidence,lift
0,Organic Cilantro,Limes,0.006,0.268,6.253
1,Limes,Large Lemon,0.009,0.203,4.328
2,Organic Lemon,Organic Hass Avocado,0.006,0.228,3.419
3,Organic Cucumber,Organic Hass Avocado,0.005,0.216,3.235
4,Organic Tomato Cluster,Organic Hass Avocado,0.004,0.208,3.115
5,Organic Blueberries,Organic Strawberries,0.008,0.241,2.993
6,Organic Raspberries,Organic Strawberries,0.010,0.239,2.963
7,Michigan Organic Kale,Organic Baby Spinach,0.004,0.212,2.946
8,Organic Small Bunch Celery,Organic Baby Spinach,0.004,0.208,2.886
9,Organic Blackberries,Organic Strawberries,0.004,0.229,2.846


## 7. Conclusion et limites

Les regles du tableau sont des pistes pour des recommandations ou des offres groupees, pas des preuves de causalite. Elles sont calculees sur un echantillon et seulement parmi les 50 produits candidats les plus frequents. Avant une recommandation commerciale, il faudra verifier les regles sur d'autres donnees, discuter leur interet metier et comparer les resultats sur un jeu de validation.

Les identifiants `product_id` sont traduits en noms grace au catalogue Instacart. Cette analyse ne relie pas encore les produits a Open Food Facts ou a RappelConso.